# Project Artifact Workflow

This notebook shows the current intended workflow for project-managed related runs:

- create a source run directly with `SimulationBase`
- capture reusable package artifacts while defining packages
- create a related run by referencing artifact ids in `package_versions`
- validate referenced artifacts before running
- let `run_simulation()` auto-apply missing artifacts
- derive a lightly modified artifact with recorded lineage


In [ ]:
from pathlib import Path
import shutil

import numpy as np

import simple_modflow as mf
from simple_modflow.modflow.mf6.simulation.discretization import DisvGrid, TemporalDiscretization
from simple_modflow.modflow.mf6.simulation.packages import CHD, InitialConditions, KFlow, OutputControl, Storage


def two_cell_vor():
    verts = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [0.0, 1.0],
            [2.0, 0.0],
            [2.0, 1.0],
        ],
        dtype=float,
    )
    iverts = [[0, 3, 2, 1], [1, 2, 5, 4]]
    xcyc = np.array([[0.5, 0.5], [1.5, 0.5]], dtype=float)
    return mf.VoronoiGridPlus(verts=verts, iverts=iverts, xcyc=xcyc)


workspace = Path("examples/mf6/artifacts/project_artifact_workflow")
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True, exist_ok=True)

catalog = mf.ProjectCatalog(workspace / "catalog", name="artifact_demo")
catalog.register_model_spec(mf.ModelSpec(name="tiny_model", grid_ref="two_cell_grid"), overwrite=True)


In [ ]:
vor = two_cell_vor()
source_model = mf.SimulationBase(
    vor=vor,
    nper=1,
    project_catalog=catalog,
    run_id="source_run",
    model_spec="tiny_model",
)
DisvGrid(vor=vor, model=source_model, top=[10.0, 10.0], bottom=[[0.0, 0.0]], nlay=1)
TemporalDiscretization(model=source_model, per_len=1, num_steps=1, multiplier=1.0)
InitialConditions(model=source_model, vor=vor, nlay=1, strt=[10.0, 8.75], artifact_id="baseline_ic")
KFlow(model=source_model, k=[1.0, 1.0], save_specific_discharge=False, artifact_id="baseline_npf")
Storage(model=source_model, sto_steady={0: True}, sto_transient={})
OutputControl(model=source_model)
CHD(
    model=source_model,
    stress_period_data={0: [[(0, 0), 10.0], [(0, 1), 8.75]]},
    artifact_id="baseline_chd",
    artifact_description="Baseline constant heads",
)
success, _ = source_model.run_simulation()
assert success is True
catalog.list_package_artifacts()


In [ ]:
related_model = mf.SimulationBase(
    vor=vor,
    nper=1,
    project_catalog=catalog,
    run_id="related_run",
    model_spec="tiny_model",
    package_versions={
        "ic": "baseline_ic",
        "npf": "baseline_npf",
        "chd": "baseline_chd",
    },
)
DisvGrid(vor=vor, model=related_model, top=[10.0, 10.0], bottom=[[0.0, 0.0]], nlay=1)
TemporalDiscretization(model=related_model, per_len=1, num_steps=1, multiplier=1.0)
Storage(model=related_model, sto_steady={0: True}, sto_transient={})
OutputControl(model=related_model)

validation_report = related_model.validate_referenced_package_artifacts()
validation_report


In [ ]:
success, _ = related_model.run_simulation()
assert success is True
related_model.summary()


In [ ]:
catalog.derive_package_artifact(
    "baseline_chd",
    artifact_id="raised_chd",
    description="Raised constant heads",
    package_data_updates={
        "stress_period_data": {
            "0": [[[0, 0], 11.0], [[0, 1], 9.25]],
        }
    },
    overwrite=True,
)

derived_model = mf.SimulationBase(
    vor=vor,
    nper=1,
    project_catalog=catalog,
    run_id="derived_run",
    model_spec="tiny_model",
    package_versions={
        "ic": "baseline_ic",
        "npf": "baseline_npf",
        "chd": "raised_chd",
    },
)
DisvGrid(vor=vor, model=derived_model, top=[10.0, 10.0], bottom=[[0.0, 0.0]], nlay=1)
TemporalDiscretization(model=derived_model, per_len=1, num_steps=1, multiplier=1.0)
Storage(model=derived_model, sto_steady={0: True}, sto_transient={})
OutputControl(model=derived_model)
success, _ = derived_model.run_simulation()
assert success is True

catalog.list_package_artifacts().set_index("artifact_id").loc[["baseline_chd", "raised_chd"]]
